# ⚡ AI Content Engine — LangGraph Pipeline
### Google Colab Notebook

This notebook walks through the **entire LangGraph pipeline** of the AI Content Engine.

**What you'll see:**
- How `PipelineState` (the shared data store) works
- Each of the 7 nodes defined and explained
- The graph wired together with `StateGraph`
- A full pipeline run that generates a **LinkedIn post**, **Twitter thread**, and **Blog post**

**Stack:** LangGraph · LangChain · Qwen2.5-72B via HuggingFace Inference Router

---
> 💡 **How to use this notebook:** Run cells top to bottom. Each section builds on the previous one.


## 📦 Step 1 — Install Dependencies

In [ ]:
# Install everything the pipeline needs
# langgraph        → the graph orchestration framework
# langchain-openai → OpenAI-compatible LLM client (works with HuggingFace too)
# huggingface-hub  → for HuggingFace API auth helpers

!pip install -q langgraph langchain-openai langchain-core huggingface-hub

## 🔑 Step 2 — Set Your HuggingFace Token

Get your token at: **https://huggingface.co/settings/tokens**
- Token type: Fine-grained
- Permission: ✅ Make calls to Inference Providers


In [ ]:
import os
from google.colab import userdata

# Load from Colab Secrets (recommended — keeps your key safe)
# Go to 🔑 Secrets in the left sidebar → add HF_TOKEN
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Colab Secrets")
except Exception:
    # Fallback: paste directly (less safe, but works for testing)
    HF_TOKEN = "hf_your_token_here"
    print("⚠️  Using hardcoded token — move to Secrets for safety")

os.environ["HF_TOKEN"] = HF_TOKEN

## 🤖 Step 3 — Configure the LLM

We use **Qwen2.5-72B-Instruct** via HuggingFace's OpenAI-compatible Inference Router.

The `ChatOpenAI` class from LangChain works perfectly here — we just swap:
- `openai_api_base` → HuggingFace's router URL
- `openai_api_key`  → your HF token


In [ ]:
from langchain_openai import ChatOpenAI

# HuggingFace Inference Router — fully OpenAI-compatible
HF_BASE_URL = "https://router.huggingface.co/v1"
MODEL       = "Qwen/Qwen2.5-72B-Instruct"

llm = ChatOpenAI(
    model           = MODEL,
    openai_api_key  = HF_TOKEN,
    openai_api_base = HF_BASE_URL,
    temperature     = 0.7,
    max_tokens      = 2500,
)

# Quick sanity check
test = llm.invoke("Say: pipeline ready")
print("LLM check:", test.content.strip())
print(f"Model: {MODEL}")

## 🗂️ Step 4 — Define PipelineState

`PipelineState` is a **TypedDict** — it's the shared memory that all 7 nodes read from and write to.

Think of it as a baton in a relay race. Each node picks it up, adds something, and passes it on.

```
Node 1 reads:  raw_notes
Node 1 writes: parsed_notes

Node 2 reads:  raw_git_log  
Node 2 writes: parsed_git

Node 3 reads:  parsed_notes + parsed_git
Node 3 writes: context
... and so on
```

`total=False` means all fields are optional — nodes only return the keys they're responsible for.


In [ ]:
from typing import TypedDict, Optional, List, Dict, Any

class PipelineState(TypedDict, total=False):
    # ── Inputs (you provide these) ──────────────────────────
    raw_notes:      str          # Your engineering notes
    raw_git_log:    str          # Git commit history
    platforms:      List[str]    # ["linkedin", "twitter", "blog"]
    author_name:    str          # Your name
    style:          str          # Creator style: "dhruv_default"
    extra_material: str          # Extra content for blog

    # ── Stage 1 & 2 outputs ─────────────────────────────────
    parsed_notes:   str          # LLM-extracted structure from notes
    parsed_git:     str          # LLM-extracted structure from git

    # ── Stage 3 output ──────────────────────────────────────
    context:        str          # Combined context document

    # ── Stage 4 outputs ─────────────────────────────────────
    narrative_angle: str         # e.g. PERFORMANCE_BREAKTHROUGH
    hook:            str         # The scroll-stopping first line
    key_detail:      str         # The credibility-building specific fact

    # ── Stage 5 output ──────────────────────────────────────
    style_guide:    str          # Loaded creator style profile

    # ── Stage 6 output ──────────────────────────────────────
    blog_blueprint: str          # Blog structure plan (stage 1 of 2)

    # ── Final output ────────────────────────────────────────
    generated_posts: Dict[str, str]  # {"linkedin": "...", "twitter": "..."}

print("✅ PipelineState defined")
print("Fields:", list(PipelineState.__annotations__.keys()))

## 📝 Step 5 — Define Prompts

These are the instructions given to the LLM at each stage.
Each prompt has `{placeholders}` filled in at runtime with actual state values.


In [ ]:
PARSE_NOTES_PROMPT = """
You are analyzing raw developer notes.
Extract structured engineering signals.

RAW NOTES:
{raw_notes}

Return EXACTLY this format:
METRICS: [numbers, latencies, percentages]
ENGINEERING_ACTIONS: [what was done]
PROBLEMS: [bugs, issues]
DECISIONS: [architecture choices]
INSIGHTS: [interesting observations]
SUMMARY: [2 sentences max]
"""

PARSE_GIT_PROMPT = """
Analyze these git commits and extract what was built.

COMMITS:
{git_log}

Return EXACTLY this format:
FEATURES: [new capabilities]
FIXES: [bugs resolved]
REFACTORING: [code improvements]
DEVELOPMENT_DIRECTION: [1 sentence — what is being built]
"""

NARRATIVE_ANGLE_PROMPT = """
You are a developer content strategist.
Choose the best narrative angle for this engineering work.

CONTEXT:
{context}

Choose ONE angle:
- DEBUGGING_STORY: A bug hunt with a surprising root cause
- PERFORMANCE_BREAKTHROUGH: A measurable improvement
- ARCHITECTURE_DECISION: A design choice that mattered
- LESSON_LEARNED: A mistake that produced real insight

Return EXACTLY this format:
ANGLE: [angle name]
HOOK: [one punchy sentence — the most surprising fact]
KEY_DETAIL: [the single most specific credibility-building detail]
"""

LINKEDIN_PROMPT = """
Write a LinkedIn post for a developer building in public.

CONTEXT: {context}
ANGLE: {angle}
HOOK: {hook}
KEY DETAIL: {key_detail}
STYLE: {style_guide}

Rules:
- First line = the hook. Make it scroll-stopping.
- Short paragraphs. First person. Real numbers.
- 160-250 words. No generic AI language.
- No "excited to share" or "learnings"

Return only the post text.
"""

TWITTER_PROMPT = """
Write a 5-tweet Twitter/X thread for developers.

CONTEXT: {context}
ANGLE: {angle}
HOOK: {hook}
KEY DETAIL: {key_detail}

Rules:
- Tweet 1 must work as a standalone hook
- Each tweet under 250 characters
- Format: 1/ text\n\n2/ text etc.
- No "Thread:" opener
"""

BLOG_BLUEPRINT_PROMPT = """
Create a blog post blueprint from this engineering content.

CONTEXT:
{context}

Return:
RECOMMENDED_TITLE: [the strongest title]
TARGET_READER: [who specifically]
MAIN_THESIS: [what the reader will learn — 1 sentence]
HOOK_PARAGRAPH: [write the actual opening paragraph]
SECTION_OUTLINE:
1. [Section] — [what it covers]
2. [Section] — [what it covers]
3. [Section] — [what it covers]
4. [Section] — [what it covers]
5. [Conclusion] — [what reader walks away with]
KEY_TECHNICAL_POINTS: [bullet list of specifics that must appear]
"""

BLOG_POST_PROMPT = """
Write a complete technical blog post.

BLUEPRINT:
{blueprint}

SOURCE MATERIAL:
{context}

Requirements:
- 1200-1500 words
- Follow the blueprint section structure
- Use markdown headers (##)
- Write like a developer explaining to another developer
- Include specific technical details from KEY_TECHNICAL_POINTS
- First person. No "In conclusion". No "Let's dive in".

Return the complete blog post in markdown.
"""

print("✅ All prompts defined")
print(f"Prompts ready: parse_notes, parse_git, narrative_angle, linkedin, twitter, blog_blueprint, blog_post")


## ⚙️ Step 6 — Define the 7 Pipeline Nodes

Each node is a **Python function** that:
1. Takes `state` (the PipelineState dict)
2. Does one job (parse, build, generate)
3. Returns a dict of **only the fields it changed**

LangGraph automatically merges the returned dict into the shared state.

---
### Node 1 — `parse_notes_node`
**Job:** Use the LLM to extract structured signals from your raw notes.


In [ ]:
from langchain_core.messages import HumanMessage

def parse_notes_node(state: PipelineState) -> dict:
    print("  → Running: parse_notes")
    raw_notes = state.get("raw_notes", "")

    if not raw_notes.strip():
        return {"parsed_notes": "SUMMARY: No notes provided."}

    prompt = PARSE_NOTES_PROMPT.format(raw_notes=raw_notes)
    response = llm.invoke([HumanMessage(content=prompt)])
    parsed = response.content.strip()

    print(f"  ✓ parse_notes done ({len(parsed)} chars)")
    return {"parsed_notes": parsed}

print("✅ Node 1: parse_notes_node defined")

### Node 2 — `parse_git_node`
**Job:** Use the LLM to extract meaning from raw git commit history.

In [ ]:
def parse_git_node(state: PipelineState) -> dict:
    print("  → Running: parse_git")
    raw_git = state.get("raw_git_log", "")

    if not raw_git.strip() or raw_git.startswith("[GIT LOG UNAVAILABLE"):
        return {"parsed_git": "FEATURES: Not available\nDEVELOPMENT_DIRECTION: Using notes only"}

    prompt = PARSE_GIT_PROMPT.format(git_log=raw_git)
    response = llm.invoke([HumanMessage(content=prompt)])
    parsed = response.content.strip()

    print(f"  ✓ parse_git done ({len(parsed)} chars)")
    return {"parsed_git": parsed}

print("✅ Node 2: parse_git_node defined")

### Node 3 — `context_builder_node`
**Job:** Merge parsed notes + parsed git into one clean context document.

⚡ **No LLM call here** — this is pure data formatting. Fast (~1ms).


In [ ]:
from datetime import datetime

def context_builder_node(state: PipelineState) -> dict:
    print("  → Running: context_builder (no LLM)")
    parsed_notes = state.get("parsed_notes", "")
    parsed_git   = state.get("parsed_git", "")
    author       = state.get("author_name", "Developer")
    platforms    = state.get("platforms", [])

    context = f"""===== ENGINEERING CONTEXT =====
Date: {datetime.now().strftime("%B %d, %Y")}
Developer: {author}
Target Platforms: {', '.join(platforms)}

--- From Developer Notes ---
{parsed_notes}

--- From Git Activity ---
{parsed_git}
================================="""

    print(f"  ✓ context_builder done ({len(context)} chars)")
    return {"context": context}

print("✅ Node 3: context_builder_node defined")

### Node 4 — `angle_node`
**Job:** Use the LLM to pick the best narrative angle, hook, and key detail.

This is what decides HOW the engineering story gets framed:
- Same latency fix → could be a `DEBUGGING_STORY` or a `PERFORMANCE_BREAKTHROUGH`
- The angle makes the difference between a boring update and a post that gets reads


In [ ]:
def angle_node(state: PipelineState) -> dict:
    print("  → Running: angle_generator")
    context = state.get("context", "")

    prompt = NARRATIVE_ANGLE_PROMPT.format(context=context)
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    # Parse the structured output line by line
    angle      = "ENGINEERING_UPDATE"
    hook       = "Here is what I built today."
    key_detail = "See context for details."

    for line in raw.split("\n"):
        line = line.strip()
        if line.upper().startswith("ANGLE:"):
            angle = line.split(":", 1)[1].strip()
        elif line.upper().startswith("HOOK:"):
            hook = line.split(":", 1)[1].strip()
        elif line.upper().startswith("KEY_DETAIL:"):
            key_detail = line.split(":", 1)[1].strip()

    print(f"  ✓ angle_generator done — angle: {angle}")
    return {
        "narrative_angle": angle,
        "hook":            hook,
        "key_detail":      key_detail,
    }

print("✅ Node 4: angle_node defined")

### Node 5 — `style_selector_node`
**Job:** Load a creator style profile that shapes the writing voice.

⚡ **No LLM call** — reads a string directly. Fast (~0ms).

In the full project this loads `.md` files from `creator_styles/`.
Here we embed the profile inline for the notebook.


In [ ]:
# In the real project, these are loaded from creator_styles/*.md files.
# Here we define them inline so the notebook is self-contained.

STYLE_PROFILES = {
    "dhruv_default": """
Style: Authentic developer voice. Honest about struggles.
Hook patterns: "I thought X. I was wrong." / "3 hours debugging. Root cause: [simple thing]."
Rhythm: Short sentences. One idea per line. No filler.
Rules: Never say "excited to share", "learnings", "game-changer".
Tone: Direct. Write like explaining to a friend in a DM, not a LinkedIn connection.
""",
    "swyx": """
Style: Reflective and analytical. Connects engineering to bigger patterns.
Hook patterns: "The real bottleneck wasn't what I expected."
Rhythm: Medium sentences. Builds an argument. Occasional rhetorical questions.
Rules: More analytical than personal. Often connects to wider developer ecosystem.
""",
    "primagen": """
Style: High energy. Direct. Opinionated. Short punchy sentences.
Hook patterns: "This broke my brain." / "Nobody talks about this and it matters."
Rhythm: Very short. Machine-gun delivery. One idea per line.
Rules: Never hedges. Picks a side. High technical credibility.
""",
}

def style_selector_node(state: PipelineState) -> dict:
    print("  → Running: style_selector (no LLM)")
    style_name = state.get("style", "dhruv_default")
    style_guide = STYLE_PROFILES.get(style_name, STYLE_PROFILES["dhruv_default"])
    print(f"  ✓ style_selector done — loaded: {style_name}")
    return {"style_guide": style_guide}

print("✅ Node 5: style_selector_node defined")

### Node 6 — `blog_blueprint_node`
**Job:** Stage 1 of two-stage blog generation.

Forces the LLM to **plan** before it writes:
- Choose the strongest title
- Define who the reader is
- Write the opening paragraph
- Outline each section's purpose

If "blog" is not in the requested platforms, this node **skips entirely** (no LLM call).


In [ ]:
def blog_blueprint_node(state: PipelineState) -> dict:
    print("  → Running: blog_blueprint")
    platforms = state.get("platforms", [])

    # Skip entirely if blog not requested — saves one full LLM call
    if "blog" not in [p.lower() for p in platforms]:
        print("  ✓ blog_blueprint skipped (blog not in platforms)")
        return {"blog_blueprint": ""}

    context = state.get("context", "")
    prompt = BLOG_BLUEPRINT_PROMPT.format(context=context)
    response = llm.invoke([HumanMessage(content=prompt)])
    blueprint = response.content.strip()

    print(f"  ✓ blog_blueprint done ({len(blueprint)} chars)")
    return {"blog_blueprint": blueprint}

print("✅ Node 6: blog_blueprint_node defined")

### Node 7 — `post_generator_node`
**Job:** The final node. Generates one post per platform using separate LLM calls.

Why separate calls per platform? LinkedIn and Twitter have completely different audience psychology. One call trying to produce both produces generic output for both. Separate, targeted prompts produce platform-native content.

For **blog**: uses the blueprint from Node 6 as a writing plan.


In [ ]:
def post_generator_node(state: PipelineState) -> dict:
    print("  → Running: post_generator")

    context    = state.get("context", "")
    angle      = state.get("narrative_angle", "ENGINEERING_UPDATE")
    hook       = state.get("hook", "Here is what I built today.")
    key_detail = state.get("key_detail", "")
    style      = state.get("style_guide", "Direct, technical, human.")
    blueprint  = state.get("blog_blueprint", "")
    platforms  = state.get("platforms", ["linkedin", "twitter"])

    generated = {}

    for platform in platforms:
        print(f"    · Generating {platform}...")

        if platform == "linkedin":
            prompt = LINKEDIN_PROMPT.format(
                context=context, angle=angle,
                hook=hook, key_detail=key_detail, style_guide=style
            )

        elif platform == "twitter":
            prompt = TWITTER_PROMPT.format(
                context=context, angle=angle,
                hook=hook, key_detail=key_detail
            )

        elif platform == "blog":
            effective_blueprint = blueprint or f"Write a technical blog post about: {context[:400]}"
            prompt = BLOG_POST_PROMPT.format(
                blueprint=effective_blueprint,
                context=context
            )

        else:
            continue

        response = llm.invoke([HumanMessage(content=prompt)])
        generated[platform] = response.content.strip()
        print(f"    ✓ {platform} done ({len(generated[platform])} chars)")

    print(f"  ✓ post_generator done — platforms: {list(generated.keys())}")
    return {"generated_posts": generated}

print("✅ Node 7: post_generator_node defined")

## 🔗 Step 7 — Assemble the LangGraph

Now we wire all 7 nodes into a `StateGraph`.

```
START
  │
  ▼
parse_notes  ──→  parse_git  ──→  context_builder
                                       │
                                  angle_generator
                                       │
                                  style_selector
                                       │
                                  blog_blueprint
                                       │
                                  post_generator
                                       │
                                      END
```

`set_entry_point` tells LangGraph where to start.
`add_edge` connects nodes in sequence.
`compile()` validates the graph and returns a runnable object.


In [ ]:
from langgraph.graph import StateGraph, END

# Create the graph with our state schema
graph = StateGraph(PipelineState)

# ── Register all 7 nodes ─────────────────────────────────
graph.add_node("parse_notes",     parse_notes_node)
graph.add_node("parse_git",       parse_git_node)
graph.add_node("context_builder", context_builder_node)
graph.add_node("angle_generator", angle_node)
graph.add_node("style_selector",  style_selector_node)
graph.add_node("blog_blueprint",  blog_blueprint_node)
graph.add_node("post_generator",  post_generator_node)

# ── Set entry point ──────────────────────────────────────
graph.set_entry_point("parse_notes")

# ── Connect nodes in order ───────────────────────────────
graph.add_edge("parse_notes",     "parse_git")
graph.add_edge("parse_git",       "context_builder")
graph.add_edge("context_builder", "angle_generator")
graph.add_edge("angle_generator", "style_selector")
graph.add_edge("style_selector",  "blog_blueprint")
graph.add_edge("blog_blueprint",  "post_generator")
graph.add_edge("post_generator",  END)

# ── Compile ──────────────────────────────────────────────
pipeline = graph.compile()

print("✅ LangGraph pipeline compiled")
print("   Nodes: 7")
print("   Entry: parse_notes")
print("   Exit:  post_generator → END")

## 👁️ Step 8 — Visualize the Graph (Optional)

LangGraph can render the graph structure as an image.


In [ ]:
try:
    from IPython.display import Image, display
    img = pipeline.get_graph().draw_mermaid_png()
    display(Image(img))
    print("✅ Graph rendered above")
except Exception as e:
    # Falls back to text if graphviz not available in environment
    print("Graph structure (text fallback):")
    print("parse_notes → parse_git → context_builder → angle_generator")
    print("           → style_selector → blog_blueprint → post_generator → END")

## 🚀 Step 9 — Define Sample Input & Run the Pipeline

This is the same input format the FastAPI endpoint accepts.
We're calling the pipeline directly here, bypassing the API layer.


In [ ]:
# Your engineering notes — write anything here
SAMPLE_NOTES = """
Worked on the voice pipeline today.

Tracked down the latency issue — finally.
Websocket buffer was holding chunks too long before forwarding.
Changed flush threshold from 4KB to 1KB.
Latency dropped: 820ms to 580ms.

VAD is still triggering on background noise (AC unit in the room).
Haven't fixed yet but I know the cause now.
Tried adjusting sensitivity — marginal improvement only.
Real fix is noise floor detection, maybe energy baseline approach.

Also committed the per-session Redis state persistence today.
Finally voice sessions survive worker restarts.
"""

SAMPLE_GIT = """
fix: websocket buffer flush threshold too high (4KB -> 1KB)
feat: add per-session redis state persistence
refactor: extract vad config to separate module
fix: race condition in audio chunk processing
"""

# Build the initial state — this is what you hand to the pipeline
initial_state = {
    "raw_notes":      SAMPLE_NOTES,
    "raw_git_log":    SAMPLE_GIT,
    "platforms":      ["linkedin", "twitter", "blog"],
    "author_name":    "Dhruv",
    "style":          "dhruv_default",
    "extra_material": "",
}

print("✅ Input ready")
print(f"   Notes:    {len(SAMPLE_NOTES)} chars")
print(f"   Git log:  {len(SAMPLE_GIT)} chars")
print(f"   Platforms: {initial_state['platforms']}")
print(f"   Style:     {initial_state['style']}")
print()
print("Ready to run. Execute the next cell to start the pipeline.")

## ▶️ Step 10 — Execute the Pipeline

`pipeline.invoke()` runs all 7 nodes in sequence and returns the final state.

Expect ~60-90 seconds total (3 platforms, ~7 LLM calls, 72B model).


In [ ]:
import time

print("=" * 55)
print("  PIPELINE STARTING")
print("=" * 55)

start = time.time()

# Run the full pipeline
# LangGraph calls each node, passes state through, merges results
final_state = pipeline.invoke(initial_state)

elapsed = round(time.time() - start, 1)

print("=" * 55)
print(f"  PIPELINE COMPLETE — {elapsed}s")
print("=" * 55)

## 🔍 Step 11 — Inspect Intermediate State

One of the best things about LangGraph: every node's output is stored in `final_state`.
You can inspect exactly what each node produced.


In [ ]:
print("━" * 55)
print("INTERMEDIATE STATE INSPECTION")
print("━" * 55)

print("\n📋 PARSED NOTES (Node 1 output):")
print("-" * 40)
print(final_state.get("parsed_notes", "not available"))

print("\n📋 PARSED GIT (Node 2 output):")
print("-" * 40)
print(final_state.get("parsed_git", "not available"))

print("\n🎯 NARRATIVE ANGLE (Node 4 output):")
print("-" * 40)
print(f"  Angle:      {final_state.get('narrative_angle', 'N/A')}")
print(f"  Hook:       {final_state.get('hook', 'N/A')}")
print(f"  Key Detail: {final_state.get('key_detail', 'N/A')}")

## 📱 Step 12 — View Generated Content

### LinkedIn Post


In [ ]:
posts = final_state.get("generated_posts", {})
linkedin = posts.get("linkedin", "Not generated")

print("━" * 55)
print("LINKEDIN POST")
print("━" * 55)
print(linkedin)
print("━" * 55)
print(f"Character count: {len(linkedin)}")
print(f"Word count:      ~{len(linkedin.split())}")

### Twitter/X Thread

In [ ]:
twitter = posts.get("twitter", "Not generated")

print("━" * 55)
print("TWITTER / X THREAD")
print("━" * 55)
print(twitter)
print("━" * 55)
print(f"Character count: {len(twitter)}")

### 📄 Blog Post

Generated via **two-stage pipeline**:
- Node 6 created the **blueprint** (title, section plan, hook paragraph)
- Node 7 wrote the **full post** against that blueprint

First, let's see the blueprint that was generated:


In [ ]:
blueprint = final_state.get("blog_blueprint", "")
print("━" * 55)
print("BLOG BLUEPRINT (Node 6 output)")
print("━" * 55)
print(blueprint[:1500])
if len(blueprint) > 1500:
    print(f"\n... [{len(blueprint) - 1500} more chars] ...")

Now the full blog post (rendered as Markdown):

In [ ]:
from IPython.display import Markdown, display

blog = posts.get("blog", "Not generated")

print(f"Word count: ~{len(blog.split())} words\n")
display(Markdown(blog))

## 📊 Step 13 — Pipeline Summary


In [ ]:
print("━" * 55)
print("PIPELINE RUN SUMMARY")
print("━" * 55)

node_descriptions = {
    "parse_notes":     "Extracted structured info from notes",
    "parse_git":       "Extracted meaning from git history",
    "context_builder": "Merged into unified context (no LLM)",
    "angle_generator": "Picked narrative angle + hook",
    "style_selector":  "Loaded creator style profile (no LLM)",
    "blog_blueprint":  "Planned blog structure",
    "post_generator":  "Generated final content per platform",
}

for i, (node, desc) in enumerate(node_descriptions.items(), 1):
    print(f"  {i}. {node:<20} → {desc}")

print()
posts_generated = final_state.get("generated_posts", {})
for platform, content in posts_generated.items():
    words = len(content.split())
    chars = len(content)
    print(f"  ✅ {platform:<12} → {words} words / {chars} chars")

print()
print(f"  Model:   {MODEL}")
print(f"  Style:   {initial_state['style']}")
print(f"  Angle:   {final_state.get('narrative_angle', 'N/A')}")
print(f"  Hook:    {final_state.get('hook', 'N/A')[:60]}...")
print()
print("━" * 55)

## 🛠️ Step 14 — Customise & Experiment

### Change the writing style


In [ ]:
# Try a different creator style and rerun
# Options: "dhruv_default", "swyx", "primagen"

custom_state = {**initial_state, "style": "primagen"}
result = pipeline.invoke(custom_state)
print(result["generated_posts"]["linkedin"])

### Generate only one platform

In [ ]:
# Only generate a Twitter thread (skips blog_blueprint entirely — saves LLM calls)
twitter_only = {**initial_state, "platforms": ["twitter"]}
result = pipeline.invoke(twitter_only)
print(result["generated_posts"]["twitter"])

### Run a single node manually

In [ ]:
# You can test any individual node without running the full graph
# Just pass a dict with the fields that node reads

test_state = {
    "raw_notes": "Fixed a memory leak in the audio processor. Was holding 200MB per session.",
    "parsed_notes": "",
}

result = parse_notes_node(test_state)
print(result["parsed_notes"])

---

## 🏗️ How This Connects to the Full Project

This notebook runs the same logic as the deployed application:

```
This notebook              Full project
─────────────────          ────────────────────────────────
pipeline.invoke()    ←→    run_pipeline_service()
PipelineState        ←→    backend/config/state.py
7 node functions     ←→    pipeline/nodes/*.py
graph.compile()      ←→    pipeline/graph.py
HuggingFace LLM      ←→    backend/llm/providers.py
STYLE_PROFILES dict  ←→    creator_styles/*.md files
```

The full project adds:
- **FastAPI** layer — serves the pipeline over HTTP
- **Cache layer** — SHA-256 hash cache, skips repeated LLM calls
- **Memory layer** — ChromaDB semantic search of past generated posts
- **Streamlit UI** — dark terminal-style frontend
- **Docker** — containerized for deployment
- **structlog** — structured JSON logging per node

The pipeline logic itself is identical.
